# Stock Span

# Problem Statement

The **stock span** for a given day is the maximum number of consecutive days ending on that day for which the stock price was less than or equal to the current day's price.

Given an array of stock prices, return the span for every day.

### Input

An array of integers:

```text
prices
```

### Output

An array containing the stock span for every day.

### Examples

```text
Input:
[100, 80, 60, 70, 60, 75, 85]

Output:
[1, 1, 1, 2, 1, 4, 6]
```

For the last day:

```text
85
```

the previous consecutive prices less than or equal to `85` are:

```text
60, 75
70, 60, 80
```

and then:

```text
100
```

is greater than `85`.

Therefore:

```text
Span = 6
```

# Problem Explanation

For each day, we need to determine:

```text
How many consecutive days immediately before today
had a price <= today's price?
```

Consider:

```text
[100, 80, 60, 70]
```

For `70`:

```text
60 <= 70
```

so we can include the previous day.

But:

```text
80 > 70
```

so we stop.

Therefore:

```text
Span of 70 = 2
```

The important observation is that we are looking for the nearest previous value that is **greater** than the current value.

That suggests:

```text
Previous Greater Element
        ↓
Monotonic Stack
```

But instead of returning the previous greater value, we need the **distance from it**.

# Brute Force

For every day, move backwards while the previous prices are less than or equal to the current price.

Example:

```text
[100, 80, 60, 70]
```

For `70`:

```text
70
↑
60 <= 70 → include
80 > 70  → stop
```

Therefore:

```text
Span = 2
```

In the worst case, an element may scan almost the entire previous portion of the array.

### Complexity

```text
Time  → O(N²)
Space → O(1)
```

A Monotonic Stack can reduce the time complexity to O(N).

# Optimal Approach

Use a Stack containing indices of prices that can still act as previous greater elements.

For each current index `i`:

While the Stack is not empty and:

```text
prices[stack[-1]] <= prices[i]
```

pop the Stack.

Why?

Because the current price is greater than or equal to that previous price, so that previous price can no longer be the nearest greater barrier for the current day or future days.

After removing all smaller or equal prices:

### If the Stack is empty

There is no previous greater price.

Therefore:

```text
span = i + 1
```

### If the Stack is not empty

The Stack top is the nearest previous greater price.

Therefore:

```text
span = i - stack[-1]
```

Then push the current index.

# Algorithm

```text
Create empty Stack
Create result array

For each index i:

    While Stack is not empty
    and price at Stack top <= current price:

        Pop Stack

    If Stack is empty:

        span = i + 1

    Else:

        span = i - Stack top

    Store span

    Push i
```

The Stack maintains indices of prices in decreasing order.

# Why Does the Stack Work?

Consider:

```text
[100, 80, 60, 70]
```

After processing:

```text
100
80
60
```

the Stack contains:

```text
[100, 80, 60]
```

Now process:

```text
70
```

`60` is not useful anymore because:

```text
60 <= 70
```

Pop it.

Now:

```text
80 > 70
```

So `80` becomes the nearest previous greater price.

Therefore:

```text
span = current_index - previous_greater_index
     = 3 - 1
     = 2
```

The Stack efficiently skips all the prices that cannot be the answer.

In [1]:
class Solution:

    def calculateSpan(self, prices: list[int]) -> list[int]:

        stack = []
        result = []

        for i in range(len(prices)):

            while stack and prices[stack[-1]] <= prices[i]:
                stack.pop()

            if not stack:
                span = i + 1
            else:
                span = i - stack[-1]

            result.append(span)
            stack.append(i)

        return result

# Dry Run

Input:

```text
[100, 80, 60, 70, 60, 75, 85]
```

Start:

```text
Stack = []
Result = []
```

### Index 0 — 100

Stack is empty.

```text
Span = 0 + 1 = 1
```

Push:

```text
Stack = [0]
```

Result:

```text
[1]
```

---

### Index 1 — 80

```text
100 > 80
```

Previous greater index:

```text
0
```

Therefore:

```text
Span = 1 - 0 = 1
```

Stack:

```text
[0, 1]
```

Result:

```text
[1, 1]
```

---

### Index 2 — 60

```text
80 > 60
```

Therefore:

```text
Span = 2 - 1 = 1
```

Stack:

```text
[0, 1, 2]
```

Result:

```text
[1, 1, 1]
```

---

### Index 3 — 70

Top is `60`.

```text
60 <= 70
```

Pop.

Now top is `80`:

```text
80 > 70
```

Therefore:

```text
Span = 3 - 1 = 2
```

Stack:

```text
[0, 1, 3]
```

Result:

```text
[1, 1, 1, 2]
```

---

### Index 4 — 60

Top is `70`.

```text
70 > 60
```

Therefore:

```text
Span = 4 - 3 = 1
```

Stack:

```text
[0, 1, 3, 4]
```

Result:

```text
[1, 1, 1, 2, 1]
```

---

### Index 5 — 75

Pop `60`:

```text
60 <= 75
```

Pop `70`:

```text
70 <= 75
```

Pop `80`:

```text
80 > 75
```

Now:

```text
Span = 5 - 1 = 4
```

Stack:

```text
[0, 1, 5]
```

Result:

```text
[1, 1, 1, 2, 1, 4]
```

---

### Index 6 — 85

Pop `75`:

```text
75 <= 85
```

Pop `80`:

```text
80 <= 85
```

Now:

```text
100 > 85
```

Therefore:

```text
Span = 6 - 0 = 6
```

Final result:

```text
[1, 1, 1, 2, 1, 4, 6]
```

# Edge Cases

### Single Price

```text
Input:
[100]

Output:
[1]
```

A single day always has span `1`.

---

### Strictly Increasing

```text
Input:
[10, 20, 30, 40]
```

Every new price is greater than all previous prices.

```text
Output:
[1, 2, 3, 4]
```

---

### Strictly Decreasing

```text
Input:
[40, 30, 20, 10]
```

Every day is blocked by the immediately previous day.

```text
Output:
[1, 1, 1, 1]
```

---

### Equal Prices

```text
Input:
[100, 100, 100]
```

Because the definition allows prices that are **less than or equal to** the current price:

```text
Output:
[1, 2, 3]
```

This is why the Stack condition uses:

```python
<=
```

# Common Mistakes

### Mistake 1 — Using `<` Instead of `<=`

The stock span includes previous prices that are equal to today's price.

Therefore:

```python
prices[stack[-1]] <= prices[i]
```

must be used.

---

### Mistake 2 — Returning the Previous Greater Value

We do not need:

```text
Previous Greater Price
```

We need:

```text
Number of consecutive days
```

Therefore calculate the index difference:

```python
i - stack[-1]
```

---

### Mistake 3 — Forgetting the Empty Stack Case

If there is no previous greater price:

```text
span = i + 1
```

because every previous day belongs to the span.

---

### Mistake 4 — Storing Prices Instead of Indices

We need the distance between the current day and the previous greater day.

Therefore the Stack must store:

```python
indices
```

not just prices.

# Complexity

Let:

```text
N = len(prices)
```

Every index is:

```text
Pushed once
Popped at most once
```

Therefore:

```text
Time → O(N)
```

The Stack can contain up to `N` indices:

```text
Space → O(N)
```

The result array requires another:

```text
O(N)
```

space.

# Why Is It O(N)?

The `while` loop may make the algorithm look like O(N²).

But each element can be popped only once.

For example:

```text
Push → 1 time
Pop  → at most 1 time
```

Across the entire array:

```text
Total pushes ≤ N
Total pops   ≤ N
```

Therefore all Stack operations together take:

```text
O(N)
```

This is the same amortized-analysis idea used in the previous Monotonic Stack problems.

# Comparison

| Approach | Time | Space |
|---|---:|---:|
| Brute Force | O(N²) | O(1) |
| Monotonic Stack | O(N) | O(N) |

The Monotonic Stack trades additional memory for linear-time processing.

# Pattern Recognition

Stock Span is essentially:

```text
Previous Greater Element
```

with one additional requirement:

```text
Calculate the distance
```

So recognize:

```text
Previous Greater
       ↓
Monotonic Stack
       ↓
Need distance
       ↓
Store indices
```

Compare this with Daily Temperatures:

```text
Next Greater
     ↓
Monotonic Stack
     ↓
Store indices
     ↓
Calculate distance
```

The direction changes, but the underlying pattern remains the same.

# Takeaway

Stock Span is a classic **Previous Greater Element** problem.

The Stack maintains useful previous prices in decreasing order.

For each current price:

```text
Remove smaller/equal prices
          ↓
Find nearest greater price
          ↓
Calculate index distance
          ↓
Push current index
```

The important connection is:

```text
Stock Span
    =
Previous Greater Element
    +
Distance
```

Complexity:

```text
Time  → O(N)
Space → O(N)
```

The larger Monotonic Stack family now includes:

```text
Next Greater Element
Next Greater Element II
Daily Temperatures
Stock Span
```